# SQL Lab Sheets 1–5 (MySQL) – Questions with Solutions

**Topics:** DDL · DML · Functions · Operators · Joins

**How to use**
1. Make sure MySQL Server is running (the same one you use in MySQL Workbench).
2. Install the connector once: `pip install mysql-connector-python pandas`
3. Edit the password in the first code cell, then run cells top to bottom.
4. Each lab sheet starts with a **setup cell** that rebuilds the tables it needs, so every sheet can also be run on its own (after Sheet 1 Q1 has created `CollegeDB`).

Requires MySQL 8.0.16+ (CHECK constraints) and 8.0.31+ for INTERSECT/EXCEPT (workarounds are given too).

## Setup: connection and helper

In [ ]:
import mysql.connector, pandas as pd, re
from IPython.display import display

conn = mysql.connector.connect(host="localhost", user="root", password="YOUR_PASSWORD")  # <-- edit
cur = conn.cursor()

def run(sql):
    """Run one or more SQL statements; show result tables for SELECT/DESC/SHOW."""
    for stmt in sql.split(";"):
        body = "\n".join(l for l in stmt.splitlines() if not l.strip().startswith("--")).strip()
        if not body:
            continue
        try:
            cur.execute(body)
            if cur.description:
                rows = cur.fetchall()
                df = pd.DataFrame(rows, columns=[d[0] for d in cur.description])
                print(">>", " ".join(body.split())[:110])
                display(df if len(df) else pd.DataFrame(columns=df.columns))
            else:
                conn.commit()
                print("OK  ", " ".join(body.split())[:110], f"({cur.rowcount} row(s) affected)")
        except mysql.connector.Error as e:
            print("ERROR:", e, "\n   in:", " ".join(body.split())[:110])
print("Connected")

---
# Lab Sheet 1: DDL Commands – CREATE, ALTER and DROP

**Q1.** Create a database named CollegeDB.

In [ ]:
run("""
DROP DATABASE IF EXISTS CollegeDB;
CREATE DATABASE CollegeDB;
USE CollegeDB;
SHOW DATABASES;
""")

**Q2.** Create a Student table with columns: Student_ID, Name, Age, Course, and City.

In [ ]:
run("""
CREATE TABLE Student (
    Student_ID INT,
    Name VARCHAR(50),
    Age INT,
    Course VARCHAR(30),
    City VARCHAR(30)
);
DESC Student;
""")

**Q3.** Create an Employee table with Emp_ID, Emp_Name, Department, and Salary.

In [ ]:
run("""
CREATE TABLE Employee (
    Emp_ID INT,
    Emp_Name VARCHAR(50),
    Department VARCHAR(30),
    Salary DECIMAL(10,2)
);
DESC Employee;
""")

**Q4.** Create a Department table with Dept_ID, Dept_Name, and Location.

In [ ]:
run("""
CREATE TABLE Department (
    Dept_ID INT,
    Dept_Name VARCHAR(50),
    Location VARCHAR(50)
);
DESC Department;
""")

**Q5.** Create a Product table with Product_ID, Product_Name, Price, and Quantity.

In [ ]:
run("""
CREATE TABLE Product (
    Product_ID INT,
    Product_Name VARCHAR(50),
    Price DECIMAL(10,2),
    Quantity INT
);
DESC Product;
""")

**Q6.** Create a Customer table with Customer_ID, Customer_Name, Email, and Phone.

In [ ]:
run("""
CREATE TABLE Customer (
    Customer_ID INT,
    Customer_Name VARCHAR(50),
    Email VARCHAR(100),
    Phone VARCHAR(15)
);
DESC Customer;
""")

**Q7.** Create a Course table with Course_ID, Course_Name, Duration, and Fees.

In [ ]:
run("""
CREATE TABLE Course (
    Course_ID INT,
    Course_Name VARCHAR(50),
    Duration VARCHAR(20),
    Fees DECIMAL(10,2)
);
DESC Course;
""")

**Q8.** Create a Faculty table with Faculty_ID, Faculty_Name, Subject, and Salary.

In [ ]:
run("""
CREATE TABLE Faculty (
    Faculty_ID INT,
    Faculty_Name VARCHAR(50),
    Subject VARCHAR(50),
    Salary DECIMAL(10,2)
);
DESC Faculty;
""")

**Q9.** Create a Library table with Book_ID, Book_Name, Author, and Price.

In [ ]:
run("""
CREATE TABLE Library (
    Book_ID INT,
    Book_Name VARCHAR(100),
    Author VARCHAR(50),
    Price DECIMAL(8,2)
);
DESC Library;
""")

**Q10.** Create a Department table with Dept_ID as the Primary Key. (The earlier table is dropped first so it can be recreated.)

In [ ]:
run("""
DROP TABLE IF EXISTS Department;
CREATE TABLE Department (
    Dept_ID INT PRIMARY KEY,
    Dept_Name VARCHAR(50),
    Location VARCHAR(50)
);
DESC Department;
""")

**Q11.** Create a Student table with Student_ID as the Primary Key and Name as NOT NULL.

In [ ]:
run("""
DROP TABLE IF EXISTS Student;
CREATE TABLE Student (
    Student_ID INT PRIMARY KEY,
    Name VARCHAR(50) NOT NULL,
    Age INT,
    Course VARCHAR(30),
    City VARCHAR(30)
);
DESC Student;
""")

**Q12.** Create an Employee table with a UNIQUE constraint on Email.

In [ ]:
run("""
DROP TABLE IF EXISTS Employee;
CREATE TABLE Employee (
    Emp_ID INT PRIMARY KEY,
    Emp_Name VARCHAR(50) NOT NULL,
    Email VARCHAR(100) UNIQUE,
    Department VARCHAR(30)
);
DESC Employee;
""")

**Q13.** Create a Product table with a CHECK constraint that Price > 0.

In [ ]:
run("""
DROP TABLE IF EXISTS Product;
CREATE TABLE Product (
    Product_ID INT PRIMARY KEY,
    Product_Name VARCHAR(50),
    Price DECIMAL(10,2) CHECK (Price > 0),
    Quantity INT
);
DESC Product;
-- Test: this insert must fail because of the CHECK constraint
INSERT INTO Product VALUES (1, 'Test', -5, 10);
""")

**Q14.** Add a new column Email to the Student table using ALTER TABLE.

In [ ]:
run("""
ALTER TABLE Student ADD Email VARCHAR(100);
DESC Student;
""")

**Q15.** Add a Phone column to the Student table.

In [ ]:
run("""
ALTER TABLE Student ADD Phone VARCHAR(15);
DESC Student;
""")

**Q16.** Modify the datatype/size of the Name column in the Student table.

In [ ]:
run("""
ALTER TABLE Student MODIFY Name VARCHAR(100) NOT NULL;
DESC Student;
""")

**Q17.** Rename the Student table to Student_Details.

In [ ]:
run("""
RENAME TABLE Student TO Student_Details;
-- (equivalent: ALTER TABLE Student RENAME TO Student_Details)
SHOW TABLES;
""")

**Q18.** Rename the column City to Address.

In [ ]:
run("""
ALTER TABLE Student_Details RENAME COLUMN City TO Address;
DESC Student_Details;
""")

**Q19.** Add a Salary column to the Employee table.

In [ ]:
run("""
ALTER TABLE Employee ADD Salary DECIMAL(10,2);
DESC Employee;
""")

**Q20.** Drop the Phone column from the Student table. (The table is now called Student_Details.)

In [ ]:
run("""
ALTER TABLE Student_Details DROP COLUMN Phone;
DESC Student_Details;
""")

**Q21.** Drop the Email column from the Student table.

In [ ]:
run("""
ALTER TABLE Student_Details DROP COLUMN Email;
DESC Student_Details;
""")

**Q22.** Remove all records from a table using TRUNCATE.

In [ ]:
run("""
INSERT INTO Student_Details (Student_ID, Name, Age, Course, Address)
VALUES (1,'Amit',20,'BCA','Meerut'), (2,'Neha',21,'MCA','Delhi');
SELECT * FROM Student_Details;
TRUNCATE TABLE Student_Details;
SELECT * FROM Student_Details;   -- empty, structure is kept
""")

**Q23.** Drop the Course table from the database.

In [ ]:
run("""
DROP TABLE Course;
SHOW TABLES;
""")

**Q24.** Drop the Customer table from the database.

In [ ]:
run("""
DROP TABLE Customer;
SHOW TABLES;
""")

**Q25.** Create a complete Student table using Primary Key, NOT NULL, UNIQUE and CHECK constraints, and demonstrate suitable ALTER and DROP operations.

In [ ]:
run("""
DROP TABLE IF EXISTS Student;
CREATE TABLE Student (
    Student_ID INT PRIMARY KEY,
    Name VARCHAR(50) NOT NULL,
    Age INT CHECK (Age >= 16),
    Email VARCHAR(100) UNIQUE,
    Course VARCHAR(30) DEFAULT 'BCA',
    City VARCHAR(30)
);
DESC Student;
-- ALTER operations
ALTER TABLE Student ADD Phone VARCHAR(15);
ALTER TABLE Student MODIFY City VARCHAR(50);
ALTER TABLE Student RENAME COLUMN City TO Address;
ALTER TABLE Student ADD CONSTRAINT chk_phone CHECK (CHAR_LENGTH(Phone) >= 10);
DESC Student;
-- DROP operations
ALTER TABLE Student DROP CHECK chk_phone;
ALTER TABLE Student DROP COLUMN Phone;
DESC Student;
DROP TABLE Student;
SHOW TABLES;
""")

---
# Lab Sheet 2: DML Commands – INSERT, UPDATE and DELETE

### Setup for Sheet 2 (recreates the tables)

In [ ]:
run("""
USE CollegeDB;
DROP TABLE IF EXISTS Student;
DROP TABLE IF EXISTS Employee;
DROP TABLE IF EXISTS Department;
DROP TABLE IF EXISTS Product;
DROP TABLE IF EXISTS Customer;
DROP TABLE IF EXISTS Course;
CREATE TABLE Student (Student_ID INT PRIMARY KEY, Name VARCHAR(50) NOT NULL, Age INT, Course VARCHAR(30), City VARCHAR(30));
CREATE TABLE Employee (Emp_ID INT PRIMARY KEY, Emp_Name VARCHAR(50), Department VARCHAR(30), Salary DECIMAL(10,2));
CREATE TABLE Department (Dept_ID INT PRIMARY KEY, Dept_Name VARCHAR(50), Location VARCHAR(50));
CREATE TABLE Product (Product_ID INT PRIMARY KEY, Product_Name VARCHAR(50), Price DECIMAL(10,2), Quantity INT);
CREATE TABLE Customer (Customer_ID INT PRIMARY KEY, Customer_Name VARCHAR(50), Email VARCHAR(100), Phone VARCHAR(15));
CREATE TABLE Course (Course_ID INT PRIMARY KEY, Course_Name VARCHAR(50), Duration VARCHAR(20), Fees DECIMAL(10,2));
SHOW TABLES;
""")

**Q1.** Insert one record into the Student table.

In [ ]:
run("""
INSERT INTO Student VALUES (1, 'Rahul', 20, 'BCA', 'Meerut');
SELECT * FROM Student;
""")

**Q2.** Insert five student records using separate INSERT statements.

In [ ]:
run("""
INSERT INTO Student VALUES (2, 'Neha', 21, 'MCA', 'Delhi');
INSERT INTO Student VALUES (3, 'Ravi', 19, 'BCA', 'Dehradun');
INSERT INTO Student VALUES (4, 'Sita', 22, 'BSc', 'Meerut');
INSERT INTO Student VALUES (5, 'Karan', 20, 'MCA', 'Lucknow');
INSERT INTO Student VALUES (6, 'Pooja', 21, 'BCom', 'Delhi');
SELECT * FROM Student;
""")

**Q3.** Insert multiple records into the Student table using a single statement.

In [ ]:
run("""
INSERT INTO Student VALUES
 (7, 'Anjali', 20, 'BCA', 'Meerut'),
 (8, 'Vikas', 23, 'MBA', 'Noida'),
 (9, 'Meena', 19, 'BSc', 'Dehradun');
SELECT * FROM Student;
""")

**Q4.** Insert records into the Employee table.

In [ ]:
run("""
INSERT INTO Employee VALUES
 (1, 'Anil', 'IT', 45000),
 (2, 'Bina', 'HR', 32000),
 (3, 'Chetan', 'IT', 52000),
 (4, 'Divya', 'Finance', 18000),
 (5, 'Esha', 'HR', 39000);
SELECT * FROM Employee;
""")

**Q5.** Insert records into the Department table.

In [ ]:
run("""
INSERT INTO Department VALUES
 (1, 'Computer Science', 'Block A'),
 (2, 'Mathematics', 'Block B'),
 (3, 'Commerce', 'Block C'),
 (4, 'Physics', 'Block D');
SELECT * FROM Department;
""")

**Q6.** Insert records into the Product table.

In [ ]:
run("""
INSERT INTO Product VALUES
 (1, 'Laptop', 55000, 5),
 (2, 'Mouse', 500, 100),
 (3, 'Keyboard', 1200, 0),
 (4, 'Monitor', 9000, 12),
 (5, 'Printer', 7500, 0);
SELECT * FROM Product;
""")

**Q7.** Insert records into the Customer table.

In [ ]:
run("""
INSERT INTO Customer VALUES
 (1, 'Arjun Mehta', 'arjun@mail.com', '9876543210'),
 (2, 'Priya Nair', 'priya@mail.com', '9123456780'),
 (3, 'Rohit Das', 'rohit@mail.com', '9988776655');
SELECT * FROM Customer;
""")

**Q8.** Insert records into the Course table.

In [ ]:
run("""
INSERT INTO Course VALUES
 (1, 'BCA', '3 Years', 60000),
 (2, 'MCA', '2 Years', 90000),
 (3, 'BSc', '3 Years', 45000);
SELECT * FROM Course;
""")

**Q9.** Insert a student record with values for all columns.

In [ ]:
run("""
INSERT INTO Student (Student_ID, Name, Age, Course, City)
VALUES (10, 'Deepak', 22, 'MBA', 'Delhi');
SELECT * FROM Student WHERE Student_ID = 10;
""")

**Q10.** Insert a student record by specifying only selected columns.

In [ ]:
run("""
INSERT INTO Student (Student_ID, Name, Age) VALUES (11, 'Farah', 20);
SELECT * FROM Student WHERE Student_ID = 11;   -- Course and City are NULL
""")

**Q11.** Update the city of a particular student.

In [ ]:
run("""
UPDATE Student SET City = 'Haridwar' WHERE Student_ID = 2;
SELECT * FROM Student WHERE Student_ID = 2;
""")

**Q12.** Update the salary of a particular employee.

In [ ]:
run("""
UPDATE Employee SET Salary = 48000 WHERE Emp_ID = 1;
SELECT * FROM Employee WHERE Emp_ID = 1;
""")

**Q13.** Increase the salary of all employees by 10%.

In [ ]:
run("""
UPDATE Employee SET Salary = Salary * 1.10;
SELECT * FROM Employee;
""")

**Q14.** Increase the price of all products by 5%.

In [ ]:
run("""
UPDATE Product SET Price = Price * 1.05;
SELECT * FROM Product;
""")

**Q15.** Change the course of a particular student.

In [ ]:
run("""
UPDATE Student SET Course = 'MCA' WHERE Student_ID = 3;
SELECT * FROM Student WHERE Student_ID = 3;
""")

**Q16.** Update the department of an employee.

In [ ]:
run("""
UPDATE Employee SET Department = 'Finance' WHERE Emp_ID = 2;
SELECT * FROM Employee WHERE Emp_ID = 2;
""")

**Q17.** Update the city of all students from Meerut to Delhi.

In [ ]:
run("""
UPDATE Student SET City = 'Delhi' WHERE City = 'Meerut';
SELECT * FROM Student;
""")

**Q18.** Update the fees of a particular course.

In [ ]:
run("""
UPDATE Course SET Fees = 65000 WHERE Course_Name = 'BCA';
SELECT * FROM Course;
""")

**Q19.** Update multiple columns of a student simultaneously.

In [ ]:
run("""
UPDATE Student SET Age = 23, Course = 'MBA', City = 'Noida' WHERE Student_ID = 5;
SELECT * FROM Student WHERE Student_ID = 5;
""")

**Q20.** Delete a particular student record using Student_ID.

In [ ]:
run("""
DELETE FROM Student WHERE Student_ID = 11;
SELECT * FROM Student;
""")

**Q21.** Delete all students belonging to a particular city.

In [ ]:
run("""
DELETE FROM Student WHERE City = 'Dehradun';
SELECT * FROM Student;
""")

**Q22.** Delete employees whose salary is below a specified amount.

In [ ]:
run("""
DELETE FROM Employee WHERE Salary < 25000;
SELECT * FROM Employee;
""")

**Q23.** Delete products whose quantity is zero.

In [ ]:
run("""
DELETE FROM Product WHERE Quantity = 0;
SELECT * FROM Product;
""")

**Q24.** Delete records satisfying multiple conditions using AND/OR.

In [ ]:
run("""
-- AND: MCA students older than 22
DELETE FROM Student WHERE Course = 'MCA' AND Age > 22;
-- OR: students from Lucknow OR studying BCom
DELETE FROM Student WHERE City = 'Lucknow' OR Course = 'BCom';
SELECT * FROM Student;
""")

**Q25.** Perform a complete sequence of INSERT, UPDATE and DELETE operations on a Student table and display the table after each operation.

In [ ]:
run("""
TRUNCATE TABLE Student;
-- INSERT
INSERT INTO Student VALUES (1,'Amit',20,'BCA','Meerut'), (2,'Neha',21,'MCA','Delhi'), (3,'Ravi',19,'BSc','Noida');
SELECT * FROM Student;
-- UPDATE
UPDATE Student SET City = 'Dehradun', Age = Age + 1 WHERE Student_ID = 1;
SELECT * FROM Student;
-- DELETE
DELETE FROM Student WHERE Student_ID = 3;
SELECT * FROM Student;
""")

---
# Lab Sheet 3: SQL Functions (MySQL)

### Setup for Sheet 3

In [ ]:
run("""
USE CollegeDB;
DROP TABLE IF EXISTS Student;
DROP TABLE IF EXISTS Employee;
CREATE TABLE Student (Student_ID INT PRIMARY KEY, First_Name VARCHAR(30), Last_Name VARCHAR(30), Age INT, City VARCHAR(30), DOB DATE);
CREATE TABLE Employee (Emp_ID INT PRIMARY KEY, Emp_Name VARCHAR(50), Department VARCHAR(30), Salary DECIMAL(10,2), Joining_Date DATE);
INSERT INTO Student VALUES
 (1,'Amit','Sharma',20,'Meerut','2005-03-14'),
 (2,'Neha','Verma',21,'Delhi','2004-08-21'),
 (3,'Ravi','Kumar',19,'Dehradun','2006-01-09'),
 (4,'Sita','Rawat',22,'Delhi','2003-11-30'),
 (5,'Karan','Singh',20,'Meerut','2005-06-17');
INSERT INTO Employee VALUES
 (1,'Anil','IT',45250.75,'2020-03-15'),
 (2,'Bina','HR',32800.40,'2019-07-01'),
 (3,'Chetan','IT',51999.99,'2021-11-20'),
 (4,'Divya','Finance',28500.50,'2022-01-10'),
 (5,'Esha','HR',39999.25,'2018-05-05');
SELECT COUNT(*) AS students FROM Student;
""")

## A. Number Functions

**Q1.** Demonstrate ABS() with suitable examples.

In [ ]:
run("""
SELECT ABS(-15) AS neg, ABS(7.5) AS pos, ABS(0) AS zero;
SELECT Student_ID, Age, ABS(Age - 25) AS Diff_From_25 FROM Student;
""")

**Q2.** Demonstrate ROUND() with suitable examples.

In [ ]:
run("""
SELECT ROUND(123.4567, 2) AS two_dec, ROUND(123.5) AS nearest_int, ROUND(1234.567, -2) AS nearest_hundred, ROUND(-2.5) AS neg;
""")

**Q3.** Demonstrate CEIL() and FLOOR().

In [ ]:
run("""
SELECT CEIL(4.2) AS ceil_pos, FLOOR(4.8) AS floor_pos, CEIL(-4.2) AS ceil_neg, FLOOR(-4.2) AS floor_neg;
""")

**Q4.** Demonstrate MOD() with suitable examples.

In [ ]:
run("""
SELECT MOD(17, 5) AS mod_fn, 17 % 5 AS mod_operator, MOD(10, 2) AS even_check;
SELECT Student_ID, IF(MOD(Student_ID, 2) = 0, 'Even', 'Odd') AS ID_Type FROM Student;
""")

**Q5.** Calculate the rounded salary of employees.

In [ ]:
run("""
SELECT Emp_Name, Salary, ROUND(Salary) AS Rounded_Salary, ROUND(Salary, -3) AS Rounded_To_Thousand FROM Employee;
""")

## B. Aggregate Functions

**Q6.** Find the total number of students using COUNT().

In [ ]:
run("""
SELECT COUNT(*) AS Total_Students FROM Student;
""")

**Q7.** Find the total salary of all employees using SUM().

In [ ]:
run("""
SELECT SUM(Salary) AS Total_Salary FROM Employee;
""")

**Q8.** Find the average salary using AVG().

In [ ]:
run("""
SELECT AVG(Salary) AS Average_Salary FROM Employee;
""")

**Q9.** Find the highest salary using MAX().

In [ ]:
run("""
SELECT MAX(Salary) AS Highest_Salary FROM Employee;
""")

**Q10.** Find the lowest salary using MIN().

In [ ]:
run("""
SELECT MIN(Salary) AS Lowest_Salary FROM Employee;
""")

## C. Character Functions

**Q11.** Demonstrate UPPER() on student names.

In [ ]:
run("""
SELECT First_Name, UPPER(First_Name) AS Upper_Name FROM Student;
""")

**Q12.** Demonstrate LOWER() on student names.

In [ ]:
run("""
SELECT First_Name, LOWER(First_Name) AS Lower_Name FROM Student;
""")

**Q13.** Demonstrate LENGTH() on student names.

In [ ]:
run("""
SELECT First_Name, LENGTH(First_Name) AS Name_Length FROM Student;
""")

**Q14.** Demonstrate CONCAT() using first and last names.

In [ ]:
run("""
SELECT CONCAT(First_Name, ' ', Last_Name) AS Full_Name FROM Student;
""")

**Q15.** Demonstrate SUBSTRING()/SUBSTR() with suitable examples.

In [ ]:
run("""
SELECT First_Name, SUBSTRING(First_Name, 1, 3) AS First_3, SUBSTR(First_Name, 2) AS From_2nd, SUBSTRING(First_Name, -2) AS Last_2 FROM Student;
""")

## D. Conversion Functions

**Q16.** Convert a number into a character/string value.

In [ ]:
run("""
SELECT CAST(12345 AS CHAR) AS num_to_str, CONCAT('Rs. ', 5000) AS implicit_conv;
SELECT Emp_Name, CONCAT('Salary is ', CAST(Salary AS CHAR)) AS Salary_Text FROM Employee;
""")

**Q17.** Convert a string into a numeric value.

In [ ]:
run("""
SELECT '123' + 0 AS implicit_num, CAST('45.50' AS DECIMAL(10,2)) AS dec_val, CAST('99' AS UNSIGNED) AS int_val;
""")

**Q18.** Convert a date value into a different format.

In [ ]:
run("""
SELECT Emp_Name, Joining_Date, DATE_FORMAT(Joining_Date, '%d-%m-%Y') AS DD_MM_YYYY, DATE_FORMAT(Joining_Date, '%W, %M %e, %Y') AS Long_Format FROM Employee;
""")

**Q19.** Demonstrate CAST() with suitable examples.

In [ ]:
run("""
SELECT CAST('2026-10-14' AS DATE) AS to_date, CAST(15.789 AS SIGNED) AS to_int, CAST(Salary AS SIGNED) AS Salary_Int FROM Employee LIMIT 3;
""")

**Q20.** Demonstrate CONVERT() where supported by the DBMS.

In [ ]:
run("""
SELECT CONVERT(123, CHAR) AS to_char, CONVERT('2026-10-14', DATE) AS to_date, CONVERT('Hello' USING utf8mb4) AS charset_conv, CONVERT(7.9, SIGNED) AS to_int;
""")

## E. Date Functions

**Q21.** Display the current date using a suitable date function.

In [ ]:
run("""
SELECT CURDATE() AS today, CURRENT_DATE() AS today2, NOW() AS date_time;
""")

**Q22.** Extract the year, month and day from a date.

In [ ]:
run("""
SELECT Emp_Name, Joining_Date, YEAR(Joining_Date) AS Yr, MONTH(Joining_Date) AS Mon, DAY(Joining_Date) AS Dy, MONTHNAME(Joining_Date) AS Month_Name FROM Employee;
""")

**Q23.** Find the difference between two dates.

In [ ]:
run("""
SELECT DATEDIFF('2026-12-31', '2026-10-14') AS Days_Diff;
SELECT Emp_Name, DATEDIFF(CURDATE(), Joining_Date) AS Days_In_Company, TIMESTAMPDIFF(YEAR, Joining_Date, CURDATE()) AS Years_In_Company FROM Employee;
""")

**Q24.** Add a specified number of days to a date.

In [ ]:
run("""
SELECT DATE_ADD('2026-10-14', INTERVAL 30 DAY) AS plus_30_days, '2026-10-14' + INTERVAL 7 DAY AS plus_7_days, DATE_SUB('2026-10-14', INTERVAL 10 DAY) AS minus_10_days;
""")

**Q25.** Display employees' joining dates in a required date format.

In [ ]:
run("""
SELECT Emp_Name, DATE_FORMAT(Joining_Date, '%d %b %Y') AS Joining_Date_Formatted FROM Employee ORDER BY Joining_Date;
""")

---
# Lab Sheet 4: SQL Operators

### Setup for Sheet 4

In [ ]:
run("""
USE CollegeDB;
DROP TABLE IF EXISTS Student;
DROP TABLE IF EXISTS Employee;
DROP TABLE IF EXISTS Product;
DROP TABLE IF EXISTS Alumni;
CREATE TABLE Student (Student_ID INT PRIMARY KEY, Name VARCHAR(50), Age INT, Course VARCHAR(30), City VARCHAR(30), Marks INT);
CREATE TABLE Employee (Emp_ID INT PRIMARY KEY, Emp_Name VARCHAR(50), Department VARCHAR(30), Salary DECIMAL(10,2), Email VARCHAR(100));
CREATE TABLE Product (Product_ID INT PRIMARY KEY, Product_Name VARCHAR(50), Price DECIMAL(10,2), Quantity INT);
CREATE TABLE Alumni (Name VARCHAR(50), City VARCHAR(30));
INSERT INTO Student VALUES
 (1,'Amit',20,'BCA','Meerut',78),(2,'Neha',21,'MCA','Delhi',85),(3,'Ravi',19,'BCA','Dehradun',66),
 (4,'Sita',22,'BSc','Delhi',91),(5,'Karan',20,'MCA','Meerut',55),(6,'Anjali',21,'BCA','Delhi',72);
INSERT INTO Employee VALUES
 (1,'Anil','IT',45000,'anil@x.com'),(2,'Bina','HR',32000,NULL),(3,'Chetan','IT',52000,'chetan@x.com'),
 (4,'Divya','Finance',28000,NULL),(5,'Esha','HR',40000,'esha@x.com'),(6,'Farhan','Sales',36000,'farhan@x.com');
INSERT INTO Product VALUES (1,'Laptop',55000,5),(2,'Mouse',500,100),(3,'Keyboard',1200,0),(4,'Monitor',9000,12),(5,'Printer',7500,3);
INSERT INTO Alumni VALUES ('Amit','Meerut'),('Zoya','Lucknow'),('Neha','Delhi');
SELECT COUNT(*) AS rows_loaded FROM Student;
""")

## Arithmetic Operators

**Q1.** Perform addition of two numbers using +.

In [ ]:
run("""
SELECT 25 + 15 AS Addition;
""")

**Q2.** Perform subtraction using -.

In [ ]:
run("""
SELECT 25 - 15 AS Subtraction;
""")

**Q3.** Perform multiplication using *.

In [ ]:
run("""
SELECT 25 * 15 AS Multiplication;
""")

**Q4.** Perform division using /.

In [ ]:
run("""
SELECT 20 / 4 AS Division, 10 / 3 AS Decimal_Division, 10 DIV 3 AS Integer_Division, 10 % 3 AS Remainder;
""")

**Q5.** Calculate the total amount as Price × Quantity.

In [ ]:
run("""
SELECT Product_Name, Price, Quantity, Price * Quantity AS Total_Amount FROM Product;
""")

## Logical Operators

**Q6.** Demonstrate AND with two conditions.

In [ ]:
run("""
SELECT * FROM Student WHERE Age >= 20 AND Marks > 70;
""")

**Q7.** Demonstrate OR with two conditions.

In [ ]:
run("""
SELECT * FROM Student WHERE City = 'Meerut' OR City = 'Dehradun';
""")

**Q8.** Demonstrate NOT with a condition.

In [ ]:
run("""
SELECT * FROM Student WHERE NOT City = 'Delhi';
""")

**Q9.** Find students belonging to a specific course AND city.

In [ ]:
run("""
SELECT * FROM Student WHERE Course = 'BCA' AND City = 'Delhi';
""")

**Q10.** Find employees belonging to either of two departments.

In [ ]:
run("""
SELECT * FROM Employee WHERE Department = 'IT' OR Department = 'HR';
""")

## Comparison Operators

**Q11.** Demonstrate the = operator.

In [ ]:
run("""
SELECT * FROM Student WHERE Course = 'MCA';
""")

**Q12.** Demonstrate the > operator.

In [ ]:
run("""
SELECT * FROM Employee WHERE Salary > 40000;
""")

**Q13.** Demonstrate the < operator.

In [ ]:
run("""
SELECT * FROM Employee WHERE Salary < 35000;
""")

**Q14.** Demonstrate >= and <=.

In [ ]:
run("""
SELECT * FROM Student WHERE Marks >= 78;
SELECT * FROM Student WHERE Marks <= 66;
""")

**Q15.** Demonstrate the <>/!= operator.

In [ ]:
run("""
SELECT * FROM Student WHERE Course <> 'BCA';
SELECT * FROM Student WHERE City != 'Delhi';
""")

## Special Operators

**Q16.** Demonstrate BETWEEN for salary range.

In [ ]:
run("""
SELECT * FROM Employee WHERE Salary BETWEEN 30000 AND 45000;
""")

**Q17.** Demonstrate IN for selecting multiple cities.

In [ ]:
run("""
SELECT * FROM Student WHERE City IN ('Delhi', 'Meerut');
""")

**Q18.** Demonstrate NOT IN.

In [ ]:
run("""
SELECT * FROM Student WHERE City NOT IN ('Delhi', 'Meerut');
""")

**Q19.** Demonstrate LIKE for names beginning with a particular letter.

In [ ]:
run("""
SELECT * FROM Student WHERE Name LIKE 'A%';      -- starts with A
SELECT * FROM Student WHERE Name LIKE '%a';      -- ends with a
SELECT * FROM Student WHERE Name LIKE '_a%';     -- 2nd letter is a
""")

**Q20.** Demonstrate IS NULL and IS NOT NULL.

In [ ]:
run("""
SELECT * FROM Employee WHERE Email IS NULL;
SELECT * FROM Employee WHERE Email IS NOT NULL;
""")

## Set Operations

**Q21.** Demonstrate UNION using two compatible queries.

In [ ]:
run("""
SELECT Name, City FROM Student UNION SELECT Name, City FROM Alumni;   -- duplicates removed
""")

**Q22.** Demonstrate UNION ALL.

In [ ]:
run("""
SELECT Name, City FROM Student UNION ALL SELECT Name, City FROM Alumni;   -- duplicates kept
""")

**Q23.** Demonstrate INTERSECT where supported by the DBMS.

In [ ]:
run("""
-- MySQL 8.0.31+
SELECT Name, City FROM Student INTERSECT SELECT Name, City FROM Alumni;
-- Workaround for older versions
SELECT DISTINCT s.Name, s.City FROM Student s WHERE (s.Name, s.City) IN (SELECT Name, City FROM Alumni);
""")

**Q24.** Demonstrate EXCEPT/MINUS where supported.

In [ ]:
run("""
-- MySQL 8.0.31+ (EXCEPT; MINUS is Oracle)
SELECT Name, City FROM Student EXCEPT SELECT Name, City FROM Alumni;
-- Workaround for older versions
SELECT Name, City FROM Student s WHERE NOT EXISTS (SELECT 1 FROM Alumni a WHERE a.Name = s.Name AND a.City = s.City);
""")

**Q25.** Perform a suitable problem using multiple operators together (Arithmetic + Logical + Comparison + Special).

In [ ]:
run("""
-- IT/HR employees with annual salary >= 4,00,000, monthly salary in 30k-60k, having an email and a name containing 'a'
SELECT Emp_Name, Department, Salary, Salary * 12 AS Annual_Salary
FROM Employee
WHERE Department IN ('IT', 'HR')
  AND Salary * 12 >= 400000
  AND Salary BETWEEN 30000 AND 60000
  AND Email IS NOT NULL
  AND Emp_Name LIKE '%a%';
""")

---
# Lab Sheet 5: SQL Joins

### Setup for Sheet 5
Department 4 (Commerce) has no students, student 5 (Karan) has no department, and student 4 (Sita) has no course – this makes outer joins meaningful.

In [ ]:
run("""
USE CollegeDB;
SET FOREIGN_KEY_CHECKS = 0;
DROP TABLE IF EXISTS Student;
DROP TABLE IF EXISTS Employee;
DROP TABLE IF EXISTS Course;
DROP TABLE IF EXISTS Department;
SET FOREIGN_KEY_CHECKS = 1;
CREATE TABLE Department (Dept_ID INT PRIMARY KEY, Dept_Name VARCHAR(50), Location VARCHAR(50));
CREATE TABLE Course (Course_ID INT PRIMARY KEY, Course_Name VARCHAR(50), Dept_ID INT, Fees DECIMAL(10,2), FOREIGN KEY (Dept_ID) REFERENCES Department(Dept_ID));
CREATE TABLE Student (Student_ID INT PRIMARY KEY, Name VARCHAR(50), Dept_ID INT, Course_ID INT,
    FOREIGN KEY (Dept_ID) REFERENCES Department(Dept_ID), FOREIGN KEY (Course_ID) REFERENCES Course(Course_ID));
CREATE TABLE Employee (Emp_ID INT PRIMARY KEY, Emp_Name VARCHAR(50), Dept_ID INT, Manager_ID INT, Salary DECIMAL(10,2),
    FOREIGN KEY (Dept_ID) REFERENCES Department(Dept_ID));
INSERT INTO Department VALUES (1,'Computer Science','Block A'),(2,'Mathematics','Block B'),(3,'Physics','Block C'),(4,'Commerce','Block D');
INSERT INTO Course VALUES (1,'BCA',1,60000),(2,'MCA',1,90000),(3,'BSc Maths',2,45000),(4,'BCom',4,40000);
INSERT INTO Student VALUES (1,'Amit',1,1),(2,'Neha',1,2),(3,'Ravi',2,3),(4,'Sita',3,NULL),(5,'Karan',NULL,NULL),(6,'Pooja',1,1);
INSERT INTO Employee VALUES (1,'Anil',1,NULL,90000),(2,'Bina',1,1,60000),(3,'Chetan',1,1,40000),(4,'Divya',2,NULL,70000),(5,'Esha',2,4,50000);
SELECT * FROM Student;
""")

**Q1.** Demonstrate an INNER JOIN between Student and Department.

In [ ]:
run("""
SELECT s.Student_ID, s.Name, d.Dept_ID, d.Dept_Name
FROM Student s INNER JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q2.** Display student names along with their department names using INNER JOIN.

In [ ]:
run("""
SELECT s.Name AS Student_Name, d.Dept_Name
FROM Student s INNER JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q3.** Display employee names along with their department names.

In [ ]:
run("""
SELECT e.Emp_Name, d.Dept_Name
FROM Employee e INNER JOIN Department d ON e.Dept_ID = d.Dept_ID;
""")

**Q4.** Display students along with their course details.

In [ ]:
run("""
SELECT s.Name, c.Course_ID, c.Course_Name, c.Fees
FROM Student s INNER JOIN Course c ON s.Course_ID = c.Course_ID;
""")

**Q5.** Find students belonging to a particular department using INNER JOIN.

In [ ]:
run("""
SELECT s.Name, d.Dept_Name
FROM Student s INNER JOIN Department d ON s.Dept_ID = d.Dept_ID
WHERE d.Dept_Name = 'Computer Science';
""")

**Q6.** Demonstrate a LEFT OUTER JOIN between Student and Department.

In [ ]:
run("""
SELECT s.Name, d.Dept_Name
FROM Student s LEFT OUTER JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q7.** Display all students, including those who do not have a matching department.

In [ ]:
run("""
SELECT s.Student_ID, s.Name, IFNULL(d.Dept_Name, 'Not Assigned') AS Dept_Name
FROM Student s LEFT JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q8.** Demonstrate a RIGHT OUTER JOIN between Student and Department.

In [ ]:
run("""
SELECT s.Name, d.Dept_Name
FROM Student s RIGHT OUTER JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q9.** Display all departments, including departments having no students.

In [ ]:
run("""
SELECT d.Dept_Name, s.Name
FROM Department d LEFT JOIN Student s ON d.Dept_ID = s.Dept_ID;
""")

**Q10.** Find students who are not assigned to any department using LEFT JOIN.

In [ ]:
run("""
SELECT s.Student_ID, s.Name
FROM Student s LEFT JOIN Department d ON s.Dept_ID = d.Dept_ID
WHERE d.Dept_ID IS NULL;
""")

**Q11.** Find departments having no students using RIGHT/LEFT JOIN.

In [ ]:
run("""
SELECT d.Dept_ID, d.Dept_Name
FROM Student s RIGHT JOIN Department d ON s.Dept_ID = d.Dept_ID
WHERE s.Student_ID IS NULL;
""")

**Q12.** Demonstrate a FULL OUTER JOIN concept using suitable queries/workaround where supported.

In [ ]:
run("""
-- MySQL has no FULL OUTER JOIN: LEFT JOIN UNION RIGHT JOIN
SELECT s.Name, d.Dept_Name FROM Student s LEFT JOIN Department d ON s.Dept_ID = d.Dept_ID
UNION
SELECT s.Name, d.Dept_Name FROM Student s RIGHT JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q13.** Demonstrate a CROSS JOIN between Student and Course.

In [ ]:
run("""
SELECT s.Name, c.Course_Name FROM Student s CROSS JOIN Course c;
""")

**Q14.** Calculate the number of possible Student-Course combinations using CROSS JOIN.

In [ ]:
run("""
SELECT COUNT(*) AS Total_Combinations FROM Student CROSS JOIN Course;   -- 6 x 4 = 24
SELECT (SELECT COUNT(*) FROM Student) * (SELECT COUNT(*) FROM Course) AS Check_Product;
""")

**Q15.** Demonstrate a NATURAL JOIN between two suitable tables.

In [ ]:
run("""
-- Student and Department share only the column Dept_ID
SELECT * FROM Student NATURAL JOIN Department;
""")

**Q16.** Compare the result of NATURAL JOIN and INNER JOIN.

In [ ]:
run("""
-- NATURAL JOIN: joins on all same-named columns automatically; the common column appears once
SELECT * FROM Student NATURAL JOIN Department;
-- INNER JOIN ... ON: you choose the condition; Dept_ID appears twice (s.Dept_ID and d.Dept_ID)
SELECT * FROM Student s INNER JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q17.** Perform a SELF JOIN on an Employee table to show employee-manager relationships.

In [ ]:
run("""
SELECT e.Emp_ID, e.Emp_Name AS Employee, m.Emp_Name AS Manager
FROM Employee e LEFT JOIN Employee m ON e.Manager_ID = m.Emp_ID;
""")

**Q18.** Display employees along with their managers using SELF JOIN.

In [ ]:
run("""
SELECT CONCAT(e.Emp_Name, ' reports to ', m.Emp_Name) AS Relationship
FROM Employee e INNER JOIN Employee m ON e.Manager_ID = m.Emp_ID;
""")

**Q19.** Join three tables—Student, Department and Course.

In [ ]:
run("""
SELECT s.Name, d.Dept_Name, c.Course_Name
FROM Student s
INNER JOIN Department d ON s.Dept_ID = d.Dept_ID
INNER JOIN Course c ON s.Course_ID = c.Course_ID;
""")

**Q20.** Display student name, course name and department name using multiple joins.

In [ ]:
run("""
-- LEFT JOINs keep students even if department/course is missing
SELECT s.Name AS Student_Name, c.Course_Name, d.Dept_Name
FROM Student s
LEFT JOIN Course c ON s.Course_ID = c.Course_ID
LEFT JOIN Department d ON s.Dept_ID = d.Dept_ID;
""")

**Q21.** Find employees whose salary is greater than the average salary of their department using joins/subqueries.

In [ ]:
run("""
SELECT e.Emp_Name, e.Dept_ID, e.Salary, a.Avg_Salary
FROM Employee e
INNER JOIN (SELECT Dept_ID, AVG(Salary) AS Avg_Salary FROM Employee GROUP BY Dept_ID) a
        ON e.Dept_ID = a.Dept_ID
WHERE e.Salary > a.Avg_Salary;
""")

**Q22.** Find the department having the maximum number of students using joins and aggregation.

In [ ]:
run("""
SELECT d.Dept_Name, COUNT(s.Student_ID) AS Total_Students
FROM Department d INNER JOIN Student s ON d.Dept_ID = s.Dept_ID
GROUP BY d.Dept_ID, d.Dept_Name
HAVING COUNT(s.Student_ID) = (
    SELECT MAX(cnt) FROM (SELECT COUNT(*) AS cnt FROM Student WHERE Dept_ID IS NOT NULL GROUP BY Dept_ID) t
);
""")

**Q23.** Display departments and the total number of students in each department.

In [ ]:
run("""
SELECT d.Dept_Name, COUNT(s.Student_ID) AS Total_Students
FROM Department d LEFT JOIN Student s ON d.Dept_ID = s.Dept_ID
GROUP BY d.Dept_ID, d.Dept_Name;
""")

**Q24.** Display courses along with the number of students enrolled in each course.

In [ ]:
run("""
SELECT c.Course_Name, COUNT(s.Student_ID) AS Students_Enrolled
FROM Course c LEFT JOIN Student s ON c.Course_ID = s.Course_ID
GROUP BY c.Course_ID, c.Course_Name;
""")

**Q25.** Create a complete query using INNER JOIN, LEFT JOIN, GROUP BY and aggregate functions to generate a student-department-course report.

In [ ]:
run("""
SELECT d.Dept_Name,
       IFNULL(c.Course_Name, 'No Course') AS Course_Name,
       COUNT(s.Student_ID) AS Total_Students,
       IFNULL(SUM(c.Fees), 0) AS Total_Fees,
       IFNULL(AVG(c.Fees), 0) AS Avg_Fees
FROM Student s
INNER JOIN Department d ON s.Dept_ID = d.Dept_ID
LEFT JOIN Course c ON s.Course_ID = c.Course_ID
GROUP BY d.Dept_Name, c.Course_Name
ORDER BY d.Dept_Name, Total_Students DESC;
""")

---
### Cleanup (optional)
Run the next cell to close the connection.

In [ ]:
cur.close()
conn.close()
print('Connection closed')